# Logistic Regression Experiment

This notebook trains and evaluates the Logistic Regression candidate using the cleaned Phase 3 splits and fitted preprocessing artifact. Hyperparameter search and cross-validation reuse `ml.training.train_models`; the test split is not used for tuning.

In [ ]:
from pathlib import Path
import sys
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

ROOT = Path.cwd()
while not (ROOT / 'ml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from ml.preprocessing.build_pipeline import PIPELINE_FILENAME
from ml.preprocessing.clean_data import TARGET_COLUMN
from ml.preprocessing.split_data import RANDOM_SEED
from ml.training.train_models import build_model_searches, _balanced_sample_weights, _classification_metrics

ARTIFACTS = ROOT / 'ml' / 'artifacts'
PROCESSED = ROOT / 'ml' / 'data' / 'processed'
sns.set_theme(style='whitegrid')
print(ROOT)

## Load data and transform features
The transformer was fitted only on the training split in Phase 3 and is reused unchanged here.

In [ ]:
preprocessor = joblib.load(ARTIFACTS / PIPELINE_FILENAME)
train = pd.read_csv(PROCESSED / 'train.csv')
validation = pd.read_csv(PROCESSED / 'validation.csv')
test = pd.read_csv(PROCESSED / 'test.csv')
x_train = preprocessor.transform(train.drop(columns=[TARGET_COLUMN]))
x_validation = preprocessor.transform(validation.drop(columns=[TARGET_COLUMN]))
x_test = preprocessor.transform(test.drop(columns=[TARGET_COLUMN]))
y_train = train[TARGET_COLUMN].astype(int)
y_validation = validation[TARGET_COLUMN].astype(int)
y_test = test[TARGET_COLUMN].astype(int)
print({'train': x_train.shape, 'validation': x_validation.shape, 'test': x_test.shape})

## Cross-validation and hyperparameter tuning
ROC-AUC is the search objective. Balanced sample weights address the 15.01% positive-class rate without changing the held-out validation or test data.

In [ ]:
search = build_model_searches(RANDOM_SEED)['logistic_regression']
search.fit(x_train, y_train, sample_weight=_balanced_sample_weights(y_train))
model = search.best_estimator_
cv_summary = pd.DataFrame({
    'best_cv_roc_auc': [search.cv_results_['mean_test_score'][search.best_index_]],
    'cv_roc_auc_std': [search.cv_results_['std_test_score'][search.best_index_]],
    'best_parameters': [search.best_params_],
})
display(cv_summary)

## Validation and test metrics
Validation metrics support model development. Test metrics are shown for a final unbiased check and are not used to tune this notebook.

In [ ]:
def metric_table(features, target, split_name):
    probabilities = model.predict_proba(features)[:, 1]
    values = _classification_metrics(target, probabilities)
    return pd.Series(values, name=split_name)

metrics = pd.concat([
    metric_table(x_validation, y_validation, 'validation'),
    metric_table(x_test, y_test, 'test'),
], axis=1)
display(metrics.round(4))

## Visual diagnostics
The plots show thresholded errors, ranking quality, and the most influential original features.

In [ ]:
validation_probabilities = model.predict_proba(x_validation)[:, 1]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay.from_predictions(y_validation, validation_probabilities >= 0.5, ax=axes[0], cmap='Blues')
axes[0].set_title('Validation confusion matrix')
RocCurveDisplay.from_predictions(y_validation, validation_probabilities, ax=axes[1])
axes[1].set_title('Validation ROC curve')
plt.tight_layout()
plt.show()

feature_metadata = json.loads((ARTIFACTS / 'feature_metadata.json').read_text())
mapping = feature_metadata['transformation_mapping']
coefficients = pd.Series(model.coef_[0], index=feature_metadata['transformed_feature_names'])
original_coefficients = coefficients.groupby(coefficients.index.map(mapping)).sum().sort_values()
top_coefficients = pd.concat([original_coefficients.head(10), original_coefficients.tail(10)])
top_coefficients.plot.barh(figsize=(9, 7), color=np.where(top_coefficients >= 0, '#c45b4d', '#3c7d8c'))
plt.title('Logistic Regression coefficient contributions by original feature')
plt.xlabel('Signed coefficient contribution')
plt.show()

## Observations
- The validation and test tables should be read together: the test split is the least biased estimate of generalization.
- Coefficients indicate model association after preprocessing, not causation.
- The probability output is not calibrated or an official credit score; calibration and model selection remain Phase 5 decisions.